<a href="https://colab.research.google.com/github/vsujancs24-max/AI-LAB/blob/main/1WN24CS312_Week9_FOL_Resolution.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
is_var = lambda t: isinstance(t, str) and t[0].islower()
fmt = lambda a: f"{a[0]}({', '.join(a[1:])})"

def unify(x, y, s=None):
    s = {} if s is None else s.copy()
    if x == y: return s
    if is_var(x): s[x] = y; return s
    if is_var(y): s[y] = x; return s
    if isinstance(x, tuple) and isinstance(y, tuple) and len(x) == len(y) and x[0] == y[0]:
        for a, b in zip(x[1:], y[1:]):
            s = unify(a, b, s)
            if s is None: return None
        return s
    return None

subst = lambda atom, s: tuple(s.get(t, t) for t in atom)

def run_pipeline(para, query):
    print("[1] PARA:\n" + para + "\n")

    sents = [x.strip() for x in para.split(".") if x.strip()]
    print("[2] SENTENCES:\n" + "\n".join(f"{i}. {s}." for i, s in enumerate(sents, 1)) + "\n")

    kb = [
        ([("Dog", "x")], ("Animal", "x")),
        ([("Animal", "x")], ("Mortal", "x")),
        ([], ("Dog", "Bruno")),
    ]
    facts = {c for pre, c in kb if not pre}

    print("[3] FOL:")
    for pre, c in kb:
        print(f"  Fact: {fmt(c)}" if not pre else f"  Rule: {' ^ '.join(fmt(p) for p in pre)} => {fmt(c)}")
    print()

    print("[4] COMBINATION ---> NEW FACTS:")
    step = 1
    while True:
        new_facts = set()
        for pre, c in kb:
            if not pre: continue
            subs = [{}]
            for p in pre:
                subs = [u for s in subs for f in facts if (u := unify(subst(p, s), f, s)) is not None]
            for s in subs:
                if (nf := subst(c, s)) not in facts | new_facts:
                    new_facts.add(nf)
                    comb = " ^ ".join(fmt(subst(p, s)) for p in pre)
                    print(f"  Step {step}: [{comb}] ---> New Fact: {fmt(nf)}")
        if not new_facts: break
        facts |= new_facts
        step += 1
    print()

    proved = any(unify(query, f) is not None for f in facts)
    print("[5] PROVE:")
    print(f"  Query : {fmt(query)} -> {'TRUE (Proved)' if proved else 'FALSE'}")

para = "All dogs are animals. All animals are mortal. Bruno is a dog."
run_pipeline(para, ("Mortal", "Bruno"))

[1] PARA:
All dogs are animals. All animals are mortal. Bruno is a dog.

[2] SENTENCES:
1. All dogs are animals.
2. All animals are mortal.
3. Bruno is a dog.

[3] FOL:
  Rule: Dog(x) => Animal(x)
  Rule: Animal(x) => Mortal(x)
  Fact: Dog(Bruno)

[4] COMBINATION ---> NEW FACTS:
  Step 1: [Dog(Bruno)] ---> New Fact: Animal(Bruno)
  Step 2: [Animal(Bruno)] ---> New Fact: Mortal(Bruno)

[5] PROVE:
  Query : Mortal(Bruno) -> TRUE (Proved)
